crop/A,B,Cから移動

In [7]:
import os
import pandas as pd
import shutil
from pathlib import Path

# --- 設定 ---
# CSVファイルのパス
csv_paths = {
    'train': '/home/hiromu/energy/src/FF/AFF/dataset_upsamp_clean/train_max_upsampled.csv',
    'val': '/home/hiromu/energy/src/FF/AFF/dataset_upsamp_clean/val_fixed.csv',
    'test': '/home/hiromu/energy/src/FF/AFF/dataset_upsamp_clean/test_fixed.csv'
}

# 【修正】画像の元フォルダのベースパスを 'combined' に変更
source_base_dir = Path('/home/hiromu/energy/data/26_0413_clean_ido/combined')

# 画像の出力先フォルダのベースパス
target_base_dir = Path('/home/hiromu/energy/data/26_0413_clean_ido/splited_dataset')

# --- 処理 ---
target_base_dir.mkdir(parents=True, exist_ok=True)

for split_name, csv_path in csv_paths.items():
    print(f"[{split_name}] の処理を開始します...")
    
    target_dir = target_base_dir / split_name
    target_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"エラー: {csv_path} が見つかりません。スキップします。")
        continue

    success_count = 0
    error_count = 0

    for index, row in df.iterrows():
        csv_filename = row['filename']
        category = row['category']
        
        # 【修正】CSVのファイル名から 'IMG_◯◯◯◯' を取り出し '_combined.png' を付ける
        base_name_parts = csv_filename.split('_')[:2] 
        base_name = '_'.join(base_name_parts)
        target_filename = f"{base_name}_combined.png"
        
        # 元画像のパス (例: .../combined/A/IMG_6420_combined.png)
        source_img_path = source_base_dir / category / target_filename
        
        # 保存先のパス (例: .../splited_dataset/train/IMG_6420_combined.png)
        target_img_path = target_dir / target_filename
        
        # ファイルのコピー
        if source_img_path.exists():
            shutil.copy(source_img_path, target_img_path)
            success_count += 1
        else:
            # もし元フォルダに本当に画像がない場合はここで出力されます
            print(f"元画像が見つかりません: {source_img_path}")
            error_count += 1

    print(f"[{split_name}] 完了: 成功 {success_count}件, 失敗(元画像なし) {error_count}件\n")

print("すべての処理が完了しました。")

[train] の処理を開始します...
[train] 完了: 成功 1062件, 失敗(元画像なし) 0件

[val] の処理を開始します...
[val] 完了: 成功 144件, 失敗(元画像なし) 0件

[test] の処理を開始します...
[test] 完了: 成功 144件, 失敗(元画像なし) 0件

すべての処理が完了しました。


CSVとフォルダで一致しているか

In [9]:
import os
import pandas as pd
from pathlib import Path

# --- 設定 ---
# 【修正】正しいパス（_cleanがついている方）に変更しました
csv_path = '/home/hiromu/energy/src/FF/AFF/dataset_upsamp_clean/test_fixed.csv'
test_dir = Path('/home/hiromu/energy/data/26_0413_clean_ido/splited_dataset/test')

# --- 処理 ---
if not test_dir.exists():
    print(f"エラー: フォルダが見つかりません。\n{test_dir}")
    exit()

existing_files = set(os.listdir(test_dir))

try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"エラー: CSVファイルが見つかりません。\n{csv_path}")
    exit()

missing_files = []

for index, row in df.iterrows():
    csv_filename = row['filename']
    
    # "IMG_66012_mask.png" から "IMG_66012_combined.png" を作成
    base_name_parts = csv_filename.split('_')[:2] 
    base_name = '_'.join(base_name_parts)
    target_filename = f"{base_name}_combined.png"

    if target_filename not in existing_files:
        missing_files.append(target_filename)

# --- 結果の出力 ---
total_csv_files = len(df)

print(f"CSV記載の総データ数: {total_csv_files} 件")
print(f"フォルダ内の実際のファイル数: {len(existing_files)} 件\n")

if not missing_files:
    print("✅ CSVに記載されているすべての画像がフォルダ内に存在します（見つからないファイルは0件です！）。")
else:
    print(f"❌ {len(missing_files)} 件の画像が見つかりませんでした。")
    for f in missing_files:
        print(f)

CSV記載の総データ数: 144 件
フォルダ内の実際のファイル数: 144 件

✅ CSVに記載されているすべての画像がフォルダ内に存在します（見つからないファイルは0件です！）。
